In [ ]:
import pandas as pd
import numpy as np
import os

from scipy.signal import find_peaks
from scipy.stats import linregress

In [ ]:
METADATA_FILE = r"C:\Users\aashi\Desktop\FYP\EDA\train.csv"

PATIENT_FOLDER = r"C:\Users\aashi\Desktop\FYP\EDA\finger tapping individual csv"

In [ ]:
metadata = pd.read_csv(METADATA_FILE)

In [ ]:
#feature extraction function
def extract_features(filepath):

    try:

        df = pd.read_csv(filepath)

        # Distance

        distance = np.sqrt(
            (df["THUMB_TIP.x"] - df["INDEX_FINGER_TIP.x"])**2 +
            (df["THUMB_TIP.y"] - df["INDEX_FINGER_TIP.y"])**2 +
            (df["THUMB_TIP.z"] - df["INDEX_FINGER_TIP.z"])**2
        )

        # Peak Detection

        peaks, _ = find_peaks(
            distance,
            prominence=0.01
        )

        if len(peaks) < 3:
            return None

        # Duration

        duration = (
            df["TIME"].max()
            - df["TIME"].min()
        )

        # Speed

        speed = len(peaks) / duration

        # Amplitude

        peak_amplitudes = distance.iloc[peaks]

        amplitude = peak_amplitudes.mean()

        # Hesitation

        peak_times = df["TIME"].iloc[peaks]

        intervals = peak_times.diff().dropna()

        hesitation = intervals.std()

        # Halt

        halt = intervals.max()

        # Decrement

        slope, _, _, _, _ = linregress(
            range(len(peak_amplitudes)),
            peak_amplitudes
        )

        return {
            "speed": speed,
            "amplitude": amplitude,
            "hesitation": hesitation,
            "halt": halt,
            "decrement": slope
        }

    except Exception as e:

        print(
            f"Error processing {filepath}"
        )

        print(e)

        return None

In [ ]:
results = []

In [ ]:
for _, row in metadata.iterrows():

    filename = row["data_file_name"]

    filepath = os.path.join(
        PATIENT_FOLDER,
        filename
    )

    if not os.path.exists(filepath):

        print(
            f"Missing: {filename}"
        )

        continue

    features = extract_features(
        filepath
    )

    if features is None:
        continue

    results.append({

        "data_file_name":
            filename,

        "updrs":
            row["doctor_diagnosis_0_5"],

        "age":
            row["age"],

        "gender":
            row["gender"],

        "patient_off_on":
            row["patient_off_on"],

        **features
    })

In [ ]:
#create feature dataset
feature_df = pd.DataFrame(results)

In [ ]:
print(feature_df.head())

print()

print(feature_df.shape)

In [ ]:
feature_df.to_csv(
    "parkinson_features.csv",
    index=False
)

print(
    "Saved successfully."
)